In [1]:
from pyspark.sql import functions as F

from utils import Gold

     
gold = Gold('GY_VALID')

df = gold.get_silver_table_df

# File 1 - Locations
pickup_locations_df = df.groupBy(F.col('PickUpLocationId').alias('LocationId'))\
.agg(
    F.sum(F.col('TotalAmount')).alias('TotalFaresPickup'), 
    F.sum(F.col('TipAmount')).cast('decimal(18,2)').alias('TotalTipAmountPickup'),
    F.avg(F.col('TripDistance')).cast('decimal(18,2)').alias('AverageDistancePickup'),
)

dropoff_locations_df = df.groupBy(F.col('DropOffLocationId').alias('LocationId'))\
.agg(
    F.avg(F.col('TripDistance')).cast('decimal(18,2)').alias('AverageDistanceDropoff'),
)

locations_df = pickup_locations_df\
.join(dropoff_locations_df, pickup_locations_df.LocationId == dropoff_locations_df.LocationId, 'full')\
.select(
    F.coalesce(pickup_locations_df.LocationId, dropoff_locations_df.LocationId).cast('int').alias('LocationId'),
    pickup_locations_df.TotalFaresPickup.alias('Total Fares by pickup location'),
    pickup_locations_df.TotalTipAmountPickup.alias('Total Tip Amount by pickup location'),
    pickup_locations_df.AverageDistancePickup.alias('The average distance by pickup location'),
    dropoff_locations_df.AverageDistanceDropoff.alias('The average distance by dropoff location'),
).orderBy(F.col('LocationId').cast('int'))

# File 2 - Vendors
vendors_df = df.groupBy(F.col('VendorId'))\
.agg(
    F.sum(F.col('TotalAmount')).cast('decimal(18,2)').alias('The total fare by vendor'), 
    F.sum(F.col('TipAmount')).cast('decimal(18,2)').alias('The total tips by vendor'),
    F.avg(F.col('TotalAmount')).cast('decimal(18,2)').alias('The average fare by vendor'),
    F.avg(F.col('TipAmount')).cast('decimal(18,2)').alias('The average tips by vendor'),
).orderBy(F.col('VendorId').cast('int'))

In [2]:
if __name__ == '__main__':
    gold.create_gold(locations_df, 'Locations')
    gold.create_gold(vendors_df, 'Vendors')

    gold.stop_spark()

--------------------------------------------------
Writing into gold: Locations
Successfully saved data in location:
./PipelineData/Gold/Locations
--------------------------------------------------
Writing into gold: Vendors
Successfully saved data in location:
./PipelineData/Gold/Vendors
